# TriDep ? 01 ? Preprocessing
Run **once** to extract all features and back them up to Drive.

Output: `/content/drive/MyDrive/DAIC/cache_backup.zip`

In [ ]:
# ============================================================
# STAGE 1 — SETUP, DRIVE MOUNT, AND MERGED LABELS (LOSOCV)
# ============================================================


from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

# ---- Paths ----
ROOT       = Path("/content/drive/MyDrive/DAIC")   # your dataset on Drive
CACHE_DIR  = Path("/content/cache")                # FAST local disk (not Drive)
CACHE_DIR.mkdir(exist_ok=True)


split_files = [
    "train_split_Depression_AVEC2017.csv",
    "dev_split_Depression_AVEC2017.csv",
    "full_test_split.csv",
]

frames = []
for f in split_files:
    p = ROOT / f
    if not p.exists():
        print("WARNING missing split file:", f)
        continue
    df = pd.read_csv(p)
    # Column names differ slightly across the 3 files; normalise them.
    cols = {c.lower(): c for c in df.columns}
    pid_col   = cols.get("participant_id")
    label_col = cols.get("phq8_binary") or cols.get("phq_binary")
    if pid_col is None or label_col is None:
        print(f"  {f}: columns found ->", df.columns.tolist())
        continue
    frames.append(df[[pid_col, label_col]].rename(
        columns={pid_col: "participant_id", label_col: "label"}))

labels = pd.concat(frames, ignore_index=True)
labels["participant_id"] = labels["participant_id"].astype(int)
labels["label"] = labels["label"].astype(int)
labels = labels.drop_duplicates("participant_id").reset_index(drop=True)

print("Total subjects with labels:", len(labels))
print("Class balance:\n", labels["label"].value_counts())
labels.to_csv(CACHE_DIR / "labels.csv", index=False)




Mounted at /content/drive
Total subjects with labels: 189
Class balance:
 label
0    133
1     56
Name: count, dtype: int64


In [ ]:
# ============================================================
# STAGE 2 — PREPROCESSING + FEATURE CACHING (RUN ONCE,
# ============================================================


import re
import librosa

SAMPLE_RATE   = 16000
N_MFCC        = 60
WINDOW_SEC    = 8
WINDOW_LEN    = SAMPLE_RATE * WINDOW_SEC   # samples per 8s window

# ---------- TEXT: clean a DAIC transcript into one text block ----------
def clean_transcript(tsv_path):
    df = pd.read_csv(tsv_path, sep="\t")
    # keep participant turns only
    df = df[df["speaker"].str.lower() == "participant"]
    lines = []
    for v in df["value"].astype(str):
        v = re.sub(r"<[^>]+>", " ", v)            # remove <laughter> etc.
        v = re.sub(r"\b(\w+)V\b", r"\1", v)        # cut-off word artefacts
        v = re.sub(r"_", " ", v)                   # l_a -> l a
        v = re.sub(r"\s+", " ", v).strip()
        if v:
            lines.append(v)
    return " ".join(lines)

# ---------- AUDIO: slice patient speech, window, MFCC + CMVN ----------
def extract_audio_segments(wav_path, tsv_path):
    y, _ = librosa.load(wav_path, sr=SAMPLE_RATE)
    df = pd.read_csv(tsv_path, sep="\t")
    df = df[df["speaker"].str.lower() == "participant"]

    # concatenate only the participant's speech
    pieces = []
    for _, row in df.iterrows():
        s = int(float(row["start_time"]) * SAMPLE_RATE)
        e = int(float(row["stop_time"])  * SAMPLE_RATE)
        if e > s:
            pieces.append(y[s:e])
    if not pieces:
        return np.empty((0, 256, N_MFCC), dtype=np.float32)
    speech = np.concatenate(pieces)

    # slice into fixed 8s windows, drop trailing short remainder
    segments = []
    for start in range(0, len(speech) - WINDOW_LEN + 1, WINDOW_LEN):
        seg = speech[start:start + WINDOW_LEN]
        mfcc = librosa.feature.mfcc(y=seg, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
        # CMVN: zero mean, unit variance per coefficient
        mfcc = (mfcc - mfcc.mean(axis=1, keepdims=True)) \
               / (mfcc.std(axis=1, keepdims=True) + 1e-8)
        segments.append(mfcc.T.astype(np.float32))   # (time, n_mfcc)
    return np.array(segments, dtype=np.float32)

# ---------- VIDEO: read FAUs, segment to match audio windows ----------
def extract_fau_segments(au_path, n_audio_segments):
    # CLNF_AUs files are comma-separated with a header; columns include
    # timestamp, confidence, success, then AU**_r (intensity) / AU**_c (presence)
    df = pd.read_csv(au_path)
    df.columns = [c.strip() for c in df.columns]
    if "success" in df.columns:
        df = df[df["success"] == 1]
    au_cols = [c for c in df.columns if c.startswith("AU")]
    arr = df[au_cols].to_numpy(dtype=np.float32)
    if len(arr) == 0 or n_audio_segments == 0:
        return np.empty((0, arr.shape[1] if arr.size else 0), dtype=np.float32)

    # normalise each AU column (zero mean, unit variance)
    arr = (arr - arr.mean(axis=0, keepdims=True)) \
          / (arr.std(axis=0, keepdims=True) + 1e-8)

    # average AU values within each window so #video == #audio segments
    chunks = np.array_split(arr, n_audio_segments)
    return np.array([c.mean(axis=0) for c in chunks], dtype=np.float32)

# ---------- main caching loop ----------
def cache_subject(pid):
    folder = ROOT / f"{pid}_P"
    wav  = folder / f"{pid}_AUDIO.wav"
    tsv  = folder / f"{pid}_TRANSCRIPT.csv"
    au   = folder / f"{pid}_CLNF_AUs.txt"
    if not (wav.exists() and tsv.exists() and au.exists()):
        print(f"  [skip {pid}] missing file(s)")
        return False

    out = CACHE_DIR / str(pid)
    out.mkdir(exist_ok=True)

    text = clean_transcript(tsv)
    (out / "text.txt").write_text(text)

    audio = extract_audio_segments(wav, tsv)
    np.save(out / "audio.npy", audio)

    fau = extract_fau_segments(au, len(audio))
    np.save(out / "fau.npy", fau)
    return True

labels = pd.read_csv(CACHE_DIR / "labels.csv")
done = 0
for pid in labels["participant_id"]:
    if cache_subject(int(pid)):
        done += 1
        if done % 10 == 0:
            print(f"cached {done} subjects...")
print(f"\nDone. Cached {done}/{len(labels)} subjects to {CACHE_DIR}")

cached 10 subjects...
cached 20 subjects...
cached 30 subjects...
cached 40 subjects...
cached 50 subjects...
cached 60 subjects...
cached 70 subjects...
cached 80 subjects...
cached 90 subjects...
cached 100 subjects...
cached 110 subjects...
cached 120 subjects...
cached 130 subjects...
cached 140 subjects...
cached 150 subjects...
cached 160 subjects...
cached 170 subjects...
cached 180 subjects...

Done. Cached 189/189 subjects to /content/cache


In [ ]:
# ============================================================
# STAGE 3 — TEXT BRANCH: SENTENCE-BERT EMBEDDINGS (FREE, NO API)
# ============================================================
# Encodes each subject's cleaned transcript into a fixed-size
# semantic vector using a pre-trained Sentence-BERT model.
# Runs on CPU fine; no payment, fully reproducible.

!pip -q install sentence-transformers

import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

CACHE_DIR = Path("/content/cache")
labels = pd.read_csv(CACHE_DIR / "labels.csv")

# nli-based SBERT model -> 768-dim sentence embeddings (Zhang et al. used SBERT)
sbert = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
EMB_DIM = sbert.get_sentence_embedding_dimension()
print("SBERT embedding dimension:", EMB_DIM)

def embed_subject(pid):
    """Read cleaned transcript, split into sentences, encode, mean-pool."""
    txt_path = CACHE_DIR / str(pid) / "text.txt"
    if not txt_path.exists():
        return None
    text = txt_path.read_text().strip()
    if not text:
        # empty transcript -> zero vector
        return np.zeros(EMB_DIM, dtype=np.float32)
    # split into sentence-like chunks; encode each; average to one vector
    sentences = [s.strip() for s in text.replace("?", ".").split(".") if s.strip()]
    if not sentences:
        sentences = [text]
    emb = sbert.encode(sentences, show_progress_bar=False, convert_to_numpy=True)
    return emb.mean(axis=0).astype(np.float32)

done = 0
for pid in labels["participant_id"]:
    vec = embed_subject(int(pid))
    if vec is None:
        print(f"  [skip {pid}] no text")
        continue
    np.save(CACHE_DIR / str(pid) / "text_emb.npy", vec)
    done += 1
    if done % 20 == 0:
        print(f"embedded {done} subjects...")

print(f"\nDone. Text embeddings saved for {done}/{len(labels)} subjects.")
print("Each subject now has: audio.npy, fau.npy, text_emb.npy")

# quick sanity check
sample = np.load(CACHE_DIR / str(labels['participant_id'].iloc[0]) / "text_emb.npy")
print("Sample text embedding shape:", sample.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_3846/869675929.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMB_DIM = sbert.get_sentence_embedding_dimension()


SBERT embedding dimension: 768
embedded 20 subjects...
embedded 40 subjects...
embedded 60 subjects...
embedded 80 subjects...
embedded 100 subjects...
embedded 120 subjects...
embedded 140 subjects...
embedded 160 subjects...
embedded 180 subjects...

Done. Text embeddings saved for 189/189 subjects.
Each subject now has: audio.npy, fau.npy, text_emb.npy
Sample text embedding shape: (768,)


In [ ]:
# ============================================================
# STAGE 3b — WAV2VEC2 AUDIO EMBEDDINGS (FREE, GPU)
# ============================================================


!pip -q install transformers torchaudio

import numpy as np
import pandas as pd
import torch, librosa
from pathlib import Path
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

ROOT      = Path("/content/drive/MyDrive/DAIC")
CACHE_DIR = Path("/content/cache")
labels    = pd.read_csv(CACHE_DIR / "labels.csv")

SAMPLE_RATE = 16000
WINDOW_SEC  = 8
WINDOW_LEN  = SAMPLE_RATE * WINDOW_SEC
MAX_WINDOWS = 20                      # cap windows per subject (speed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL = "facebook/wav2vec2-base-960h"
fe    = Wav2Vec2FeatureExtractor.from_pretrained(MODEL)
w2v   = Wav2Vec2Model.from_pretrained(MODEL).to(device).eval()
EMB   = w2v.config.hidden_size        # 768

@torch.no_grad()
def embed_windows(windows):
    """windows: list of 1-D np arrays (each WINDOW_LEN samples) -> (n,768)"""
    inputs = fe(windows, sampling_rate=SAMPLE_RATE,
                return_tensors="pt", padding=True)
    iv = inputs.input_values.to(device)
    out = w2v(iv).last_hidden_state        # (n, frames, 768)
    return out.mean(dim=1).cpu().numpy()   # mean-pool time -> (n, 768)

def patient_windows(pid):
    folder = ROOT / f"{pid}_P"
    wav = folder / f"{pid}_AUDIO.wav"
    tsv = folder / f"{pid}_TRANSCRIPT.csv"
    if not (wav.exists() and tsv.exists()):
        return []
    y, _ = librosa.load(wav, sr=SAMPLE_RATE)
    df = pd.read_csv(tsv, sep="\t")
    df = df[df["speaker"].str.lower() == "participant"]
    pieces = []
    for _, r in df.iterrows():
        s = int(float(r["start_time"]) * SAMPLE_RATE)
        e = int(float(r["stop_time"])  * SAMPLE_RATE)
        if e > s:
            pieces.append(y[s:e])
    if not pieces:
        return []
    speech = np.concatenate(pieces).astype(np.float32)
    wins = [speech[i:i+WINDOW_LEN]
            for i in range(0, len(speech) - WINDOW_LEN + 1, WINDOW_LEN)]
    return wins[:MAX_WINDOWS]

done = 0
for pid in labels["participant_id"]:
    pid = int(pid)
    wins = patient_windows(pid)
    if not wins:
        vec = np.zeros(EMB * 2, dtype=np.float32)
    else:
        emb = embed_windows(wins)                       # (n, 768)
        vec = np.concatenate([emb.mean(axis=0),
                              emb.std(axis=0)]).astype(np.float32)  # (1536,)
    np.save(CACHE_DIR / str(pid) / "wav2vec.npy", vec)
    done += 1
    if done % 20 == 0:
        print(f"  wav2vec embedded {done}/{len(labels)}...")

print(f"\nDone. wav2vec.npy saved for {done} subjects. dim = {EMB*2}")

# ---- back up the enriched cache to Drive (verified) ----
import shutil, zipfile, time
shutil.make_archive("/content/cache_backup", "zip", str(CACHE_DIR))
shutil.copy("/content/cache_backup.zip",
            "/content/drive/MyDrive/DAIC/cache_backup.zip")
time.sleep(5)
print("cache backup valid on Drive:",
      zipfile.is_zipfile("/content/drive/MyDrive/DAIC/cache_backup.zip"))

device: cuda


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  wav2vec embedded 20/189...
  wav2vec embedded 40/189...
  wav2vec embedded 60/189...
  wav2vec embedded 80/189...
  wav2vec embedded 100/189...
  wav2vec embedded 120/189...
  wav2vec embedded 140/189...
  wav2vec embedded 160/189...
  wav2vec embedded 180/189...

Done. wav2vec.npy saved for 189 subjects. dim = 1536
cache backup valid on Drive: True


In [ ]:
import shutil, zipfile, time
from pathlib import Path

CACHE_DIR   = Path("/content/cache")
DRIVE_ZIP   = "/content/drive/MyDrive/DAIC/cache_backup.zip"

# write to LOCAL disk first (fast, reliable), then copy to Drive
local_zip = "/content/cache_backup.zip"
shutil.make_archive("/content/cache_backup", "zip", str(CACHE_DIR))
print("local zip valid:", zipfile.is_zipfile(local_zip))

shutil.copy(local_zip, DRIVE_ZIP)
time.sleep(5)  # give Drive a moment

# verify it landed on Drive
p = Path(DRIVE_ZIP)
print("on Drive:", p.exists(), "| size MB:", round(p.stat().st_size/1e6,1) if p.exists() else "N/A")
print("valid zip on Drive:", zipfile.is_zipfile(p) if p.exists() else "N/A")

local zip valid: True
on Drive: True | size MB: 610.1
valid zip on Drive: True


In [ ]:
import numpy as np
print(np.load("/content/cache/301/text_emb.npy").shape)  # expect (768,)

(768,)
